# LoRA vs QLoRA Fine-Tuning of a 1B LLaMA-based Instruction Model

This notebook fine-tunes **TinyLlama/TinyLlama-1.1B-Chat-v1.0** (a 1.1B
parameter, LLaMA-architecture, instruction-tuned model) using:

1. **LoRA** -- base weights frozen, fp16 base model, low-rank adapters on
   `q_proj` and `v_proj` only.
2. **QLoRA** -- base weights frozen and loaded in **4-bit NF4** quantized
   format via `bitsandbytes`, with the same LoRA adapter configuration.

For each method we record: number of trainable parameters, final training
loss, and peak GPU memory usage. We then generate outputs for the same 3
test prompts under each fine-tuned model and compare them qualitatively.

**Why TinyLlama-1.1B-Chat instead of a gated Llama checkpoint?** It is
openly licensed (Apache 2.0, no gated access request / HF token approval
needed), genuinely LLaMA-architecture, ~1.1B parameters, and already
instruction-tuned -- so it fits the assignment's "1B LLaMA-based
instruction model" requirement without an access-approval delay that
would block running this in a single Colab session. If you have approved
access to `meta-llama/Llama-3.2-1B-Instruct` instead, you can swap
`MODEL_NAME` below and everything else works unchanged.

**Runtime**: In Colab, set **Runtime -> Change runtime type -> T4 GPU**
before running. QLoRA's 4-bit loading requires a CUDA GPU (bitsandbytes
does not support 4-bit quantization on CPU); if you're using the
"alternate local environment" fallback from the assignment's submission
note, you'll need a CUDA-capable GPU locally for the QLoRA section to run
as written.


In [ ]:
# 1. Install dependencies
# (Colab already has torch; pinning the rest to versions known to work
# together as of this notebook's authoring. If a newer bitsandbytes/peft
# release changes an API used below, re-pin to these versions.)
!pip install -q -U "transformers>=4.44,<4.47" "peft>=0.12,<0.14" \
    "bitsandbytes>=0.43.1" "accelerate>=0.33" "datasets>=2.20"


In [ ]:
# 2. Imports and GPU check
import gc
import time
import json

import torch
import pandas as pd
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

assert torch.cuda.is_available(), (
    "No CUDA GPU detected. In Colab: Runtime -> Change runtime type -> T4 GPU. "
    "QLoRA's 4-bit quantization requires a CUDA GPU."
)
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


## Model, dataset, and shared hyperparameters

To keep this reproducible and fast on a free Colab GPU (and to avoid any
external dataset flakiness), we use a small, self-contained instruction
dataset embedded directly in the notebook. Swap `TRAIN_EXAMPLES` for a
real dataset (e.g. `databricks/databricks-dolly-15k` via
`datasets.load_dataset`) for a more meaningful fine-tuning result -- the
rest of the pipeline is unaffected either way.


In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Shared hyperparameters (identical for both LoRA and QLoRA runs, per the
# assignment: the point of the comparison is memory/behavior differences
# from quantization, not from differing training settings).
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]
NUM_EPOCHS = 2
LEARNING_RATE = 2e-4
MAX_SEQ_LEN = 256
PER_DEVICE_BATCH_SIZE = 4

TEST_PROMPTS = [
    "Explain what a black hole is in simple terms.",
    "Write a short, polite email asking a colleague to review a document.",
    "What are three tips for staying focused while studying?",
]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


In [ ]:
# A small, self-contained instruction dataset (instruction -> response).
# Kept short on purpose so 2 epochs finish quickly on a free-tier GPU.
TRAIN_EXAMPLES = [
    {"instruction": "What is the capital of France?", "response": "The capital of France is Paris."},
    {"instruction": "Summarize the water cycle in one sentence.", "response": "The water cycle is the continuous movement of water through evaporation, condensation, and precipitation between the Earth's surface and atmosphere."},
    {"instruction": "Give me a healthy breakfast idea.", "response": "Try Greek yogurt topped with berries, a drizzle of honey, and a handful of granola for a balanced, protein-rich breakfast."},
    {"instruction": "Explain photosynthesis briefly.", "response": "Photosynthesis is the process by which plants use sunlight, water, and carbon dioxide to produce glucose and oxygen."},
    {"instruction": "What is 15% of 200?", "response": "15% of 200 is 30."},
    {"instruction": "Write a one-sentence motivational quote about persistence.", "response": "Success is rarely a straight line, but it always rewards those who keep walking."},
    {"instruction": "Name three renewable energy sources.", "response": "Three renewable energy sources are solar power, wind power, and hydroelectric power."},
    {"instruction": "How do I politely decline a meeting invite?", "response": "Thank you for the invite, but I won't be able to attend this time. Please keep me posted on any outcomes or follow-up actions."},
    {"instruction": "What causes seasons on Earth?", "response": "Seasons are caused by the tilt of the Earth's axis as it orbits the Sun, which changes the angle and intensity of sunlight different regions receive throughout the year."},
    {"instruction": "Give a simple definition of machine learning.", "response": "Machine learning is a field of AI where computer systems learn patterns from data to make predictions or decisions without being explicitly programmed for each task."},
    {"instruction": "Suggest a good icebreaker question for a team meeting.", "response": "Ask: 'What's one small thing that made you smile this week?' It's light, inclusive, and easy for anyone to answer."},
    {"instruction": "What's the difference between weather and climate?", "response": "Weather refers to short-term atmospheric conditions in a specific place, while climate describes long-term average weather patterns over a larger region and time span."},
    {"instruction": "Give me a simple recipe for scrambled eggs.", "response": "Whisk 2 eggs with a pinch of salt, pour into a heated, buttered non-stick pan, and stir gently over low heat until softly set."},
    {"instruction": "Explain why the sky is blue.", "response": "The sky appears blue because molecules in the atmosphere scatter shorter blue wavelengths of sunlight more than other colors, spreading blue light across the sky."},
    {"instruction": "What is compound interest?", "response": "Compound interest is interest calculated on both the initial principal and the accumulated interest from previous periods, causing savings or debt to grow faster over time."},
    {"instruction": "Give three tips for better sleep.", "response": "Keep a consistent sleep schedule, avoid screens for an hour before bed, and keep your bedroom cool and dark."},
]

print(f"Training examples: {len(TRAIN_EXAMPLES)}")


In [ ]:
def format_prompt(instruction: str) -> str:
    """Uses the tokenizer's own chat template so formatting matches
    whatever base model is plugged in (not hardcoded to one model's
    special tokens)."""
    messages = [
        {"role": "system", "content": "You are a helpful, concise assistant."},
        {"role": "user", "content": instruction},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def tokenize_example(example: dict) -> dict:
    """Tokenizes prompt+response as one sequence, masking the prompt
    tokens (and padding) out of the loss with label = -100 so the model
    is only trained to predict the response, not to reproduce the prompt."""
    prompt = format_prompt(example["instruction"])
    full_text = prompt + example["response"] + tokenizer.eos_token

    tokenized_full = tokenizer(
        full_text, truncation=True, max_length=MAX_SEQ_LEN, padding="max_length"
    )
    prompt_len = len(tokenizer(prompt, truncation=True, max_length=MAX_SEQ_LEN)["input_ids"])

    labels = list(tokenized_full["input_ids"])
    for i in range(len(labels)):
        is_padding = tokenized_full["attention_mask"][i] == 0
        is_prompt = i < prompt_len
        if is_padding or is_prompt:
            labels[i] = -100

    tokenized_full["labels"] = labels
    return tokenized_full


tokenized_dataset = [tokenize_example(ex) for ex in TRAIN_EXAMPLES]
print("Example tokenized keys:", list(tokenized_dataset[0].keys()))
print("Sequence length:", len(tokenized_dataset[0]["input_ids"]))


## Shared utilities

These helpers are used identically for both the LoRA and the QLoRA run so
the *only* difference between the two experiments is how the base model
is loaded (fp16 vs 4-bit NF4) -- everything else (data, LoRA config,
training hyperparameters, measurement method) is held constant, which is
what makes the comparison meaningful.


In [ ]:
class ListDataset(torch.utils.data.Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        item = self.examples[idx]
        return {k: torch.tensor(v) for k, v in item.items()}


train_dataset = ListDataset(tokenized_dataset)


def count_trainable_parameters(model) -> dict:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return {
        "trainable_params": trainable,
        "total_params": total,
        "pct_trainable": 100 * trainable / total,
    }


def reset_gpu_memory_stats():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


def get_peak_gpu_memory_gb() -> float:
    return torch.cuda.max_memory_allocated() / (1024 ** 3)


def run_training(model, run_name: str, output_dir: str):
    """Trains `model` on train_dataset for NUM_EPOCHS and returns a dict of
    the metrics the assignment asks us to record."""
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        logging_steps=1,
        save_strategy="no",
        report_to=[],
        bf16=(COMPUTE_DTYPE == torch.bfloat16),
        fp16=(COMPUTE_DTYPE == torch.float16),
        run_name=run_name,
    )

    trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset)

    start = time.time()
    train_result = trainer.train()
    elapsed = time.time() - start

    final_loss = trainer.state.log_history[-1].get("train_loss")
    if final_loss is None:
        step_losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
        final_loss = step_losses[-1] if step_losses else None

    peak_mem_gb = get_peak_gpu_memory_gb()
    param_stats = count_trainable_parameters(model)

    return {
        "run_name": run_name,
        "final_train_loss": final_loss,
        "peak_gpu_memory_gb": round(peak_mem_gb, 3),
        "train_runtime_sec": round(elapsed, 1),
        **param_stats,
    }


@torch.no_grad()
def generate_responses(model, prompts):
    model.eval()
    outputs = {}
    for prompt in prompts:
        formatted = format_prompt(prompt)
        inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
        output_ids = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
        generated_text = tokenizer.decode(
            output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )
        outputs[prompt] = generated_text.strip()
    return outputs


## Part A -- LoRA fine-tuning

Base model loaded in fp16 with **all weights frozen**; LoRA adapters
(`r=8`) are added only to `q_proj` and `v_proj` in the attention layers.


In [ ]:
reset_gpu_memory_stats()

base_model_lora = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=COMPUTE_DTYPE,
    device_map="auto",
)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

lora_model = get_peft_model(base_model_lora, lora_config)
lora_model.print_trainable_parameters()  # sanity check: base weights should be frozen


In [ ]:
lora_results = run_training(lora_model, run_name="lora", output_dir="./lora-out")
print(json.dumps(lora_results, indent=2))


In [ ]:
lora_outputs = generate_responses(lora_model, TEST_PROMPTS)
for prompt, response in lora_outputs.items():
    print(f"PROMPT: {prompt}\nLoRA RESPONSE: {response}\n{'-'*80}")


In [ ]:
# Save the LoRA adapter, then free GPU memory before the QLoRA run so peak
# memory measurements for QLoRA aren't inflated by leftover LoRA tensors.
lora_model.save_pretrained("./lora-adapter")

del lora_model
del base_model_lora
gc.collect()
torch.cuda.empty_cache()


## Part B -- QLoRA fine-tuning

Same LoRA configuration (`r=8`, `q_proj`/`v_proj`, same epochs/learning
rate), but the base model is now loaded in **4-bit NF4** quantized format
via `bitsandbytes` before adapters are attached.


In [ ]:
reset_gpu_memory_stats()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

base_model_qlora = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

# Required prep step for k-bit (4-bit/8-bit) training: casts norm layers to
# fp32 for stability, enables gradient-checkpointing-compatible inputs, etc.
base_model_qlora = prepare_model_for_kbit_training(base_model_qlora)

qlora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

qlora_model = get_peft_model(base_model_qlora, qlora_config)
qlora_model.print_trainable_parameters()


In [ ]:
qlora_results = run_training(qlora_model, run_name="qlora", output_dir="./qlora-out")
print(json.dumps(qlora_results, indent=2))


In [ ]:
qlora_outputs = generate_responses(qlora_model, TEST_PROMPTS)
for prompt, response in qlora_outputs.items():
    print(f"PROMPT: {prompt}\nQLoRA RESPONSE: {response}\n{'-'*80}")


In [ ]:
qlora_model.save_pretrained("./qlora-adapter")


## Results comparison


In [ ]:
comparison_df = pd.DataFrame([lora_results, qlora_results]).set_index("run_name")
comparison_df = comparison_df[
    ["trainable_params", "total_params", "pct_trainable",
     "final_train_loss", "peak_gpu_memory_gb", "train_runtime_sec"]
]
comparison_df


In [ ]:
mem_reduction_pct = 100 * (1 - qlora_results["peak_gpu_memory_gb"] / lora_results["peak_gpu_memory_gb"])
print(f"QLoRA peak GPU memory vs LoRA: {mem_reduction_pct:.1f}% lower")
print(f"Trainable params identical? {lora_results['trainable_params'] == qlora_results['trainable_params']}")


## Qualitative comparison of generated outputs

The cell below prints both models' responses to the same 3 prompts side
by side. **After running this notebook, fill in your own observations**
in the markdown cell that follows -- this depends on the actual numbers
your run produces, so it's left as a template rather than a canned
conclusion.


In [ ]:
for prompt in TEST_PROMPTS:
    print(f"PROMPT: {prompt}\n")
    print(f"  LoRA : {lora_outputs[prompt]}\n")
    print(f"  QLoRA: {qlora_outputs[prompt]}\n")
    print("=" * 100)


### Observations (fill in after running)

Use the printed comparison table and generations above to answer:

- **Trainable parameters**: Were they identical between LoRA and QLoRA?
  (They should be -- quantization changes how the *frozen* base is stored,
  not how many adapter parameters you train.)
- **Peak GPU memory**: By how much did 4-bit loading reduce peak memory?
  Was the reduction close to what you'd expect from going fp16/bf16 (2
  bytes/param) to 4-bit (~0.5 bytes/param) for the ~1.1B frozen base
  parameters?
- **Final training loss**: Was it similar between the two runs, or did
  quantization noise measurably change convergence given the same
  hyperparameters?
- **Generation quality**: For each of the 3 prompts, do the LoRA and
  QLoRA outputs differ meaningfully in coherence, relevance, or style? Is
  either one clearly better, or are they comparable given how small/short
  this fine-tuning run was?
- **Practical takeaway**: Given the memory savings observed, when would
  you choose QLoRA over LoRA for a larger model where full fp16 fine-tuning
  wouldn't fit on your available GPU at all?
